In [ ]:


from pathlib import Path
import os
import json
import time

import pandas as pd
from tqdm.auto import tqdm

from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env
load_dotenv()

# Create OpenAI client 
api_key = os.getenv("OPENAI_API_KEY") or os.getenv("OPENAI_API_KEY1")
client = OpenAI(api_key=api_key)


# Project paths
PROJECT_ROOT = Path("..").resolve()
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Processed data dir:", DATA_PROCESSED)


Project root: C:\Users\User\Documents\retailmind-prototype
Processed data dir: C:\Users\User\Documents\retailmind-prototype\data\processed


In [11]:
# Load subset data
IN_PATH = DATA_PROCESSED / "uss_english_turns_llm_satisfaction_subset.parquet"  # demo output
OUT_PATH = DATA_PROCESSED / "uss_english_issue_tagged_llm_subset.parquet"

# Load turns with LLM satisfaction + last_system_text
df = pd.read_parquet(IN_PATH)
print("Loaded:", IN_PATH, df.shape)
df.head(2)
print(df["speaker"].value_counts())


Loaded: C:\Users\User\Documents\retailmind-prototype\data\processed\uss_english_turns_llm_satisfaction_subset.parquet (39102, 21)
speaker
USER      20693
SYSTEM    18409
Name: count, dtype: int64


In [12]:
# Select low-satisfaction USER turns 
mask_low = (
    (df["speaker"] == "USER")
    & (df.get("is_overall", False) == False)
    & (df.get("low_satisfaction_llm", False) == True)
)

df_low = df.loc[mask_low].copy()
print("Low-satisfaction USER turns selected:", len(df_low))

df_low[["dataset","conv_id","turn_id","text","llm_satisfaction_score","llm_satisfaction_reason"]].head(5)


Low-satisfaction USER turns selected: 506


,dataset,conv_id,turn_id,text,llm_satisfaction_score,llm_satisfaction_reason
91,CCPE,2,33,"Oh, I dislike it. It's I don't know if it's of...",2.0,User expresses strong dislike and frustration ...
351,CCPE,13,22,"So, I would say I'm kind of in the middle on t...",2.0,"User expresses a negative opinion, indicating ..."
464,CCPE,18,10,I found the movie to be just too over the top ...,2.0,User expresses dissatisfaction with the movie ...
489,CCPE,19,16,It's just boring boring to me.,2.0,User expresses dissatisfaction and finds roman...
637,CCPE,25,19,"Nope, haven't seen that.",2.0,User expresses dissatisfaction by rejecting th...


In [13]:

# df_low inherits last_system_text from df
df_low["last_system_text"] = df_low["last_system_text"].fillna("[No previous system turn available]")


In [14]:
# Build snippet: what the bot said + user's message + satisfaction hint
df_low["snippet"] = (
    "Bot: " + df_low["last_system_text"].astype(str) + "\n"
    "User: " + df_low["text"].astype(str) + "\n"
    "Satisfaction score (estimated): " + df_low["llm_satisfaction_score"].astype(str) + "\n"
    "Satisfaction rationale: " + df_low["llm_satisfaction_reason"].fillna("").astype(str)
)

df_low[["dataset", "conv_id", "turn_id", "snippet"]].head()


,dataset,conv_id,turn_id,snippet
91,CCPE,2,33,Bot: Do you like or dislike the movie Aliens?\...
351,CCPE,13,22,"Bot: did you like it?\nUser: So, I would say I..."
464,CCPE,18,10,Bot: What about that kind of movie didn't you ...
489,CCPE,19,16,Bot: what do you dislike about romance movies?...
637,CCPE,25,19,"Bot: How about the Incredibles 2\nUser: Nope, ..."


In [15]:
MAX_TAG_ROWS = 800  # demo-friendly; increase later if needed

if len(df_low) > MAX_TAG_ROWS:
    df_low_run = df_low.sample(n=MAX_TAG_ROWS, random_state=42).copy()
else:
    df_low_run = df_low.copy()

print("Rows actually sent to LLM tagger:", len(df_low_run))


Rows actually sent to LLM tagger: 506


In [16]:
# Taxonomy of issue labels
ISSUE_LABELS = [
    "MISSING_CONTEXT",
    "WRONG_FACT",
    "TONE_ISSUE",
    "LOOP",
    "UNSUPPORTED_INTENT",
    "SLOW_RESPONSE",
    "HANDOFF_REQUIRED",
    "SUCCESS_BEST_PRACTICE",
]

def build_issue_prompt(snippet: str) -> str:
    """
    Build the prompt given a conversation snippet.
    """
    taxonomy_str = ", ".join(ISSUE_LABELS)
    return f"""
You are an assistant that diagnoses why a chatbot reply led to low user satisfaction.

You receive a short conversation snippet (last bot message and user follow-up).
Your job:
1. Decide which issue categories apply, using this taxonomy:
   {taxonomy_str}
2. Explain briefly why (1-3 sentences).
3. Estimate severity: LOW, MEDIUM, or HIGH.

Guidelines:
- Use MISSING_CONTEXT when the bot ignores prior information or asks for data it already has.
- Use WRONG_FACT when the bot gives incorrect or misleading information.
- Use TONE_ISSUE when the bot is rude, robotic, or lacks empathy while the user is upset.
- Use LOOP when the bot repeats itself or gets stuck.
- Use UNSUPPORTED_INTENT when the user asks for something outside the bot's capabilities.
- Use SLOW_RESPONSE when the delay/latency clearly bothers the user.
- Use HANDOFF_REQUIRED when a human agent should take over.
- If there is no real failure, use SUCCESS_BEST_PRACTICE as the only issue.

Output a JSON object with exactly this schema:
{{
  "issues": ["ISSUE_1", "ISSUE_2", ...],  # at least one issue; use "SUCCESS_BEST_PRACTICE" if no real failure
  "severity": "LOW" | "MEDIUM" | "HIGH",
  "reason": "short explanation here"
}}

Conversation snippet:
---
{snippet}
---
"""


In [17]:
MODEL_NAME = "gpt-4o-mini"  #  try models later
TEMPERATURE = 0.1

def call_issue_tagger(snippet: str, max_retries: int = 3, sleep_seconds: float = 2.0) -> dict:
    """
    Call OpenAI to tag issues for the given snippet.
    Returns a dict: {"issues": [...], "severity": "...", "reason": "..."}.
    
    """
    prompt = build_issue_prompt(snippet)

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                temperature=TEMPERATURE,
                response_format={"type": "json_object"},
                messages=[
                    {
                        "role": "system",
                        "content": "You are an AI assistant specialized in diagnosing chatbot failures. Always output valid JSON."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    },
                ],
            )

            content = response.choices[0].message.content
            data = json.loads(content)

            # Defaults
            data.setdefault("issues", [])
            data.setdefault("severity", "MEDIUM")
            data.setdefault("reason", "")

            # Normalize issues to the allowed labels
            normalized_issues = []
            for issue in data.get("issues", []):
                issue = str(issue).strip().upper()
                if issue in ISSUE_LABELS:
                    normalized_issues.append(issue)

            # If LLM didn't pick any valid label, fall back
            if not normalized_issues:
                normalized_issues = ["SUCCESS_BEST_PRACTICE"]

            data["issues"] = normalized_issues

            # Normalize severity
            sev = str(data.get("severity", "MEDIUM")).upper()
            if sev not in {"LOW", "MEDIUM", "HIGH"}:
                sev = "MEDIUM"
            data["severity"] = sev

            # Reason as string
            data["reason"] = str(data.get("reason", "")).strip()

            return data

        except Exception as e:
            print(f"Error on attempt {attempt + 1}: {e}")
            time.sleep(sleep_seconds)

    # Fallback if all retries fail
    return {
        "issues": ["HANDOFF_REQUIRED"],
        "severity": "HIGH",
        "reason": "Tagging failed due to API error.",
    }


In [18]:
issue_results = []

# iterrows with tqdm
for idx, row in tqdm(df_low_run.iterrows(), total=len(df_low_run)):
    snippet = row["snippet"]
    result = call_issue_tagger(snippet)

    issue_results.append({
        "dataset": row["dataset"],
        "conv_id": row["conv_id"],
        "turn_id": row["turn_id"],
        "issues": result["issues"],
        "severity": result["severity"],
        "reason": result["reason"],
    })

len(issue_results)


100%|██████████| 506/506 [13:33<00:00,  1.61s/it]


506

In [19]:
df_issues = pd.DataFrame(issue_results)
df_issues.head()


,dataset,conv_id,turn_id,issues,severity,reason
0,CCPE,2,33,"[MISSING_CONTEXT, TONE_ISSUE]",MEDIUM,The bot fails to acknowledge the user's strong...
1,CCPE,13,22,[MISSING_CONTEXT],MEDIUM,The bot's question about whether the user like...
2,CCPE,18,10,"[MISSING_CONTEXT, TONE_ISSUE]",MEDIUM,The bot fails to acknowledge the user's negati...
3,CCPE,19,16,[MISSING_CONTEXT],MEDIUM,The bot's question about what the user dislike...
4,CCPE,25,19,[MISSING_CONTEXT],MEDIUM,The bot fails to acknowledge the user's previo...


In [22]:
print("df_issues shape:", df_issues.shape)
print("Null checks:")
print(df_issues.isna().sum())

# Ensure IDs exist and are unique per turn
dups = df_issues.duplicated(subset=["dataset", "conv_id", "turn_id"]).sum()
print("Duplicate (dataset, conv_id, turn_id) rows:", dups)

# Quick label sanity
print("\nExample issues field types:", df_issues["issues"].apply(type).value_counts)


df_issues shape: (506, 6)
Null checks:
dataset     0
conv_id     0
turn_id     0
issues      0
severity    0
reason      0
dtype: int64
Duplicate (dataset, conv_id, turn_id) rows: 0

Example issues field types: <bound method IndexOpsMixin.value_counts of 0      <class 'list'>
1      <class 'list'>
2      <class 'list'>
3      <class 'list'>
4      <class 'list'>
            ...      
501    <class 'list'>
502    <class 'list'>
503    <class 'list'>
504    <class 'list'>
505    <class 'list'>
Name: issues, Length: 506, dtype: object>


In [24]:
print("Does df already have issues/severity/reason?",
      [c for c in ["issues","severity","reason"] if c in df.columns])

tmp = df.merge(df_issues, on=["dataset","conv_id","turn_id"], how="left")
print("\nMerged columns containing 'issue':", [c for c in tmp.columns if "issue" in c.lower()])
print("Merged columns containing 'severity':", [c for c in tmp.columns if "severity" in c.lower()])
print("Merged columns containing 'reason':", [c for c in tmp.columns if "reason" in c.lower()])

tmp.head(1)


Does df already have issues/severity/reason? ['issues', 'severity', 'reason']

Merged columns containing 'issue': ['issues_x', 'issues_y']
Merged columns containing 'severity': ['severity_x', 'severity_y']
Merged columns containing 'reason': ['reason_x', 'llm_satisfaction_reason', 'reason_y']


,conv_id,turn_id,speaker,text,dialog_act,scores_raw,scores,is_overall,satisfaction_mean,label,...,severity_x,reason_x,system_text_only,last_system_text,llm_satisfaction_score,llm_satisfaction_reason,low_satisfaction_llm,issues_y,severity_y,reason_y
0,0,1,SYSTEM,Do you like movies like Thor?,None,None,None,False,NaN,ENTITY_NAME+MOVIE_OR_SERIES,...,NONE,,Do you like movies like Thor?,Do you like movies like Thor?,NaN,None,False,NaN,NaN,NaN


In [25]:
df_tagged = df.merge(
    df_issues,
    on=["dataset", "conv_id", "turn_id"],
    how="left",
    suffixes=("", "_tag")  # columns from df_issues become issues_tag, severity_tag, reason_tag if conflicts
)

# If df already had issues/severity/reason, they will remain as-is and new ones are *_tag.
# If df did not have them, then issues_tag may not exist; handle both cases.

if "issues_tag" in df_tagged.columns:
    df_tagged["issues"] = df_tagged["issues_tag"]
else:
    # If no conflict, df_issues columns may have been merged directly as 'issues'
    df_tagged["issues"] = df_tagged.get("issues", np.nan)

if "severity_tag" in df_tagged.columns:
    df_tagged["severity"] = df_tagged["severity_tag"]
else:
    df_tagged["severity"] = df_tagged.get("severity", np.nan)

if "reason_tag" in df_tagged.columns:
    df_tagged["reason"] = df_tagged["reason_tag"]
else:
    df_tagged["reason"] = df_tagged.get("reason", np.nan)

# Drop helper cols if they exist
df_tagged = df_tagged.drop(columns=[c for c in ["issues_tag","severity_tag","reason_tag"] if c in df_tagged.columns])

# Fill defaults for non-tagged rows
df_tagged["issues"] = df_tagged["issues"].apply(lambda x: x if isinstance(x, list) else [])
df_tagged["severity"] = df_tagged["severity"].fillna("NONE")
df_tagged["reason"] = df_tagged["reason"].fillna("")

print("Rows in df_tagged:", len(df_tagged))
print("Turns with >=1 issue:", (df_tagged["issues"].apply(len) > 0).sum())
print("\nSeverity counts:")
print(df_tagged["severity"].value_counts())


Rows in df_tagged: 39102
Turns with >=1 issue: 506

Severity counts:
severity
NONE      38596
HIGH        277
MEDIUM      228
LOW           1
Name: count, dtype: int64


In [26]:
# The 506 tagged turns should be a subset of turns with issues length > 0
tagged_keys = set(zip(df_issues["dataset"], df_issues["conv_id"], df_issues["turn_id"]))
merged_keys = set(zip(
    df_tagged.loc[df_tagged["issues"].apply(len) > 0, "dataset"],
    df_tagged.loc[df_tagged["issues"].apply(len) > 0, "conv_id"],
    df_tagged.loc[df_tagged["issues"].apply(len) > 0, "turn_id"],
))

print("Tagged keys:", len(tagged_keys))
print("Keys with issues after merge:", len(merged_keys))
print("Tagged keys missing after merge:", len(tagged_keys - merged_keys))


Tagged keys: 506
Keys with issues after merge: 506
Tagged keys missing after merge: 0


In [28]:
out_path = DATA_PROCESSED / "uss_english_issue_tagged_llm_subset.parquet"
df_tagged.to_parquet(out_path, index=False)
print("Saved issue-tagged dataset to:", out_path)


Saved issue-tagged dataset to: C:\Users\User\Documents\retailmind-prototype\data\processed\uss_english_issue_tagged_llm_subset.parquet


In [29]:
from collections import Counter
from itertools import chain

issue_counter = Counter(chain.from_iterable(df_tagged["issues"]))
print("Top issue labels:", issue_counter.most_common(10))

print("\nSeverity counts:")
print(df_tagged["severity"].value_counts())

print("\nDistinct issue labels:", sorted(issue_counter.keys()))


Top issue labels: [('MISSING_CONTEXT', 422), ('UNSUPPORTED_INTENT', 209), ('TONE_ISSUE', 59), ('WRONG_FACT', 18), ('HANDOFF_REQUIRED', 2), ('SUCCESS_BEST_PRACTICE', 1)]

Severity counts:
severity
NONE      38596
HIGH        277
MEDIUM      228
LOW           1
Name: count, dtype: int64

Distinct issue labels: ['HANDOFF_REQUIRED', 'MISSING_CONTEXT', 'SUCCESS_BEST_PRACTICE', 'TONE_ISSUE', 'UNSUPPORTED_INTENT', 'WRONG_FACT']
